# fy-m1 04 — synthetic everyday chats (T4 x2)
Few-shot prompts an open-weight model with the real, anonymised chats in the private `Fan1000-Prog/fyM1-chats` and
keeps only chats that pass `fym1.data.synth.filter_conversations` (Malagasy assistant turns, length bounds, no
placeholders or unknown names after anonymising, no 8-word run shared with any real chat, no duplicates).
**Runtime → Run all.** Raw outputs go to `fyM1-chats` `synth/raw/part-<first index>.ndjson` every `UPLOAD_EVERY`
chats, so a rerun after a disconnect resumes where the last part ended. Each run then filters everything generated
so far and writes `synth/filtered.ndjson` and counts-only `synth/synth_stats.json`
(fetch it with `scripts/build_synth_stats.py`). Nothing here prints chat text.

`.ndjson`, not `.jsonl`: `01_train` trains on every `*.jsonl` in the repo, so these files stay out of training until
the sample check; then `PROMOTE = True` writes `synth/synth_chats.jsonl`.
Generator: the first entry of `GENERATORS` that loads fully on the GPUs and passes a one-chat smoke check; a part
that hits a float16 inf/nan moves to the next entry and is redone (the stats record which generators ran, with their
licences). Generators are permissive only (owner decision 2026-10-18: fy-m1 ships under a simple open licence):
`synth.generator_licence` refuses any other model, or a model card whose licence changed, before anything loads.
Kaggle: Accelerator **GPU T4 x2**, Internet on, `HF_TOKEN` secret on (no gated-model licence to accept). Code comes from `fyM1-ckpt` `code/` (`scripts/sync_code_to_hub.py`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U transformers accelerate bitsandbytes fasttext-numpy2-wheel

In [ ]:
GENERATORS = [  # (hub id, dtype), tried in order; permissive licences only (synth.GENERATOR_LICENCES)
    # float16 is ~2x faster on a T4 (no bfloat16) and ran the 9B's baseline; float32 avoids float16 inf/nan.
    # Gemma 4 runs float32: on a T4 Unsloth keeps gemma4 in float32 (00_baseline, 01_train).
    ("McGill-NLP/AfriqueQwen3.5-9B-50Langs-Instruct-v1", "float16"),
    ("McGill-NLP/AfriqueQwen3.5-9B-50Langs-Instruct-v1", "float32"),
    ("google/gemma-4-12B-it", "float32"),
]
TARGET_RAW = 2500         # raw chats to generate; the goal is >= 2,000 kept after filtering
FEW_SHOT_K, SEED = 3, 0   # real examples per prompt; SEED fixes the seed grid, example picks and sampling
GEN_BATCH, MAX_NEW_TOKENS, TEMPERATURE, TOP_P = 8, 400, 0.9, 0.95
UPLOAD_EVERY = 48         # chats per raw part on the Hub: a disconnect loses at most this many
MAX_HOURS = 11.0          # stop generating after this (Kaggle sessions end at 12 h), then filter and save stats
WEIGHTS_GIB_PER_GPU = 11  # weights cap per T4 (15 GB); the rest is left for the KV cache
REAL_FILES = ["chats.jsonl"]  # real chats in fyM1-chats: few-shot examples and the 8-word leak reference
PROMOTE = False           # True only after the sample check: also writes synth/synth_chats.jsonl (01_train input)
CKPT_REPO, DATA_REPO = "Fan1000-Prog/fyM1-ckpt", "Fan1000-Prog/fyM1-chats"
RAW_PREFIX = "synth/raw/"

In [ ]:
# Runtime bootstrap (Colab or Kaggle). The HF token is needed before fym1 can be downloaded, so this cell
# carries a verbatim copy of fym1.runtime.hf_token and its helpers (checked by tests/test_notebook_runtime.py).
import os
import tempfile

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"  # both of Kaggle's T4s: a 27B model in 4-bit needs ~17 GB; a single-GPU runtime just sees GPU 0
CODE_DIR = tempfile.mkdtemp(prefix="fym1code-")  # fresh each run: nothing stale on sys.path

MISSING_TOKEN = ("no HF_TOKEN found: on Kaggle attach it under Add-ons → Secrets; on Colab add it in the "
                 "Secrets pane with notebook access; elsewhere set the HF_TOKEN environment variable")


def on_kaggle() -> bool:
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
        return True
    try:
        import kaggle_secrets  # noqa: F401  (only installed in Kaggle kernels)
    except ImportError:
        return False
    return True


def _secret(get) -> str | None:
    try:
        return get("HF_TOKEN")
    except Exception:  # missing or not attached: the caller raises one clear error; never echo details
        return None


def hf_token() -> str:
    """HF token from Kaggle secrets, else Colab secrets, else the HF_TOKEN env var. Never print it."""
    if on_kaggle():
        from kaggle_secrets import UserSecretsClient

        token = _secret(UserSecretsClient().get_secret)
    else:
        try:
            from google.colab import userdata
        except ImportError:
            token = os.environ.get("HF_TOKEN")
        else:
            token = _secret(userdata.get)
    if not token:
        raise RuntimeError(MISSING_TOKEN)
    return token

In [ ]:
import sys, json, time, datetime
from huggingface_hub import HfApi, hf_hub_download, list_repo_files, snapshot_download

SESSION_START = time.time()
os.environ["HF_TOKEN"] = hf_token()  # never print it
api = HfApi()
code_dir = snapshot_download(CKPT_REPO, allow_patterns=["code/**"], local_dir=CODE_DIR)
sys.path.insert(0, f"{code_dir}/code/src")
CODE_SHA = open(f"{code_dir}/code/SHA").read().strip()
from fym1.data import synth
from fym1.eval.batching import run_batched
from fym1.runtime import output_dir, scratch_dir
from fym1.train.chat_format import CHAT_TEMPLATE_KWARGS

OUTPUT, SCRATCH = output_dir(), scratch_dir()
DATA_REVISION = api.dataset_info(DATA_REPO).sha
real = []
for f in REAL_FILES:  # private chat text: kept in memory only, never printed or saved here
    path = hf_hub_download(DATA_REPO, f, repo_type="dataset", revision=DATA_REVISION)
    real += [json.loads(l) for l in open(path) if l.strip()]
seeds = synth.seed_grid(TARGET_RAW, SEED)


def raw_rows():
    files = [f for f in list_repo_files(DATA_REPO, repo_type="dataset") if f.startswith(RAW_PREFIX)]
    if not files:
        return []
    local = snapshot_download(DATA_REPO, repo_type="dataset", allow_patterns=[f"{RAW_PREFIX}*"],
                              local_dir=f"{SCRATCH}/synth")
    return [json.loads(l) for f in sorted(files) for l in open(f"{local}/{f}") if l.strip()]


LICENCES = {g: synth.generator_licence(g, card_licence=getattr(api.model_info(g).card_data, "license", None))
            for g, _ in GENERATORS}  # refuses a non-permissive generator before anything runs
existing = raw_rows()
for g in sorted({row["generator"] for row in existing}):  # parts from an earlier, refused generator
    synth.generator_licence(g)  # raises: delete synth/raw/ in fyM1-chats and rerun
done = {row["index"] for row in existing}
todo = [i for i in range(TARGET_RAW) if i not in done]
print("real chats:", len(real), "| raw done:", len(done), "| to do:", len(todo), "| code", CODE_SHA)

In [ ]:
import gc
import torch
from transformers import AutoModelForImageTextToText, AutoTokenizer, BitsAndBytesConfig

GPUS = torch.cuda.device_count()


def load(hub_id, dtype):
    torch_dtype = getattr(torch, dtype)
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                               bnb_4bit_compute_dtype=torch_dtype)
    model = AutoModelForImageTextToText.from_pretrained(
        hub_id, quantization_config=quant, torch_dtype=torch_dtype, device_map="auto",
        max_memory={i: f"{WEIGHTS_GIB_PER_GPU}GiB" for i in range(GPUS)})
    off_gpu = {d for d in model.hf_device_map.values() if not isinstance(d, int)}
    if off_gpu:  # CPU/disk offload would be far too slow: treat as "does not fit"
        raise MemoryError(f"needs offload to {sorted(map(str, off_gpu))}")
    tok = AutoTokenizer.from_pretrained(hub_id)
    tok.padding_side = "left"
    return model.eval(), tok


def prompt_text(i):
    examples = [r["messages"] for r in synth.examples_for(i, real, FEW_SHOT_K, SEED)]
    chat = [{"role": "user", "content": synth.build_prompt(seeds[i], examples)}]
    return tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True,
                                   **CHAT_TEMPLATE_KWARGS)  # thinking off: Qwen3.5 and Gemma 4 can think


@torch.inference_mode()
def generate(indices):
    """(output text, new tokens) per seed index; batches that run out of memory are retried one by one."""
    def run(batch):
        enc = tok([prompt_text(i) for i in batch], return_tensors="pt", padding=True,
                  add_special_tokens=False).to(model.device)
        ids = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=True, temperature=TEMPERATURE,
                             top_p=TOP_P, pad_token_id=tok.pad_token_id)
        new = ids[:, enc["input_ids"].shape[1]:]
        counts = (new != tok.pad_token_id).sum(dim=1).tolist()
        return list(zip(tok.batch_decode(new, skip_special_tokens=True), counts))

    return run_batched(indices, GEN_BATCH, run, is_oom=lambda e: isinstance(e, torch.OutOfMemoryError),
                       clear=torch.cuda.empty_cache)


def free_model():
    global model, tok
    model = tok = None
    gc.collect()
    torch.cuda.empty_cache()


def load_next_generator():
    """Load the next GENERATORS entry that fits on the GPUs and passes a smoke check; raise when none is left."""
    global model, tok, GEN_ID, DTYPE, NEXT_GENERATOR
    while NEXT_GENERATOR < len(GENERATORS):
        GEN_ID, DTYPE = GENERATORS[NEXT_GENERATOR]
        NEXT_GENERATOR += 1
        try:
            model, tok = load(GEN_ID, DTYPE)
            torch.manual_seed(SEED)
            smoke = generate(todo[:2])  # two chats, so one unlucky sample does not skip a good model
            if any(synth.parse_conversation(text) for text, _ in smoke):  # outputs never printed
                print("GENERATOR", GEN_ID, DTYPE, LICENCES[GEN_ID], "| skipped:", LOAD_LOG,
                      "| VRAM GB", [round(torch.cuda.memory_allocated(i) / 2**30, 1) for i in range(GPUS)])
                return
            LOAD_LOG.append(f"{GEN_ID} {DTYPE}: smoke output did not parse")
        except Exception as e:  # does not fit, float16 overflow (inf/nan in sampling), ...
            LOAD_LOG.append(f"{GEN_ID} {DTYPE}: {type(e).__name__}: {str(e)[:200]}")
        free_model()
    raise RuntimeError(f"no generator left: {LOAD_LOG}")


model = tok = None
LOAD_LOG = []  # counts-only reasons a generator was skipped
NEXT_GENERATOR = 0  # index of the next GENERATORS entry to try
GPU_NAMES = f"{GPUS}x {torch.cuda.get_device_name(0)}"
if todo:
    load_next_generator()

In [ ]:
start = 0
while start < len(todo):
    if time.time() - SESSION_START > MAX_HOURS * 3600:
        print("time budget reached; filtering what exists")
        break
    part = todo[start:start + UPLOAD_EVERY]
    torch.manual_seed(SEED * 1_000_003 + part[0])  # same part, same sampling on a rerun
    t0 = time.time()
    try:
        results = generate(part)
    except RuntimeError as e:
        if not synth.is_sampling_overflow(e):
            raise
        LOAD_LOG.append(f"{GEN_ID} {DTYPE}: inf/nan in sampling at part {part[0]:05d}")
        free_model()
        load_next_generator()  # then redo this part
        continue
    seconds = (time.time() - t0) / len(part)
    rows = [{"index": i, "seed": seeds[i].as_dict(), "generator": GEN_ID, "dtype": DTYPE,
             "licence": LICENCES[GEN_ID], "code_sha": CODE_SHA, "data_revision": DATA_REVISION, "output": text,
             "new_tokens": n, "gen_seconds": round(seconds, 2)}
            for i, (text, n) in zip(part, results)]
    body = "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows).encode()
    api.upload_file(path_or_fileobj=body, path_in_repo=f"{RAW_PREFIX}part-{part[0]:05d}.ndjson", repo_id=DATA_REPO,
                    repo_type="dataset", commit_message=f"synth raw {part[0]}-{part[-1]}")
    print(f"part {part[0]:05d}: {len(rows)} chats, {sum(r['new_tokens'] for r in rows)} new tokens, "
          f"{seconds:.1f} s/chat | {len(done) + start + len(part)}/{TARGET_RAW}")
    start += UPLOAD_EVERY

In [ ]:
from collections import Counter
from fym1.eval.lang import is_malagasy

rows = sorted(raw_rows(), key=lambda r: r["index"])
kept, dropped = synth.filter_conversations((synth.parse_conversation(r["output"]) for r in rows), real,
                                           is_malagasy=is_malagasy)
generators = sorted({(r["generator"], r["dtype"]) for r in rows})
stats = synth.synth_stats(
    raw=len(rows), kept=len(kept), dropped=dropped, by_generator=Counter(r["generator"] for r in rows),
    meta={"seed": SEED, "few_shot_k": FEW_SHOT_K, "target_raw": TARGET_RAW,
          "generator": ", ".join(g for g, _ in generators), "dtype": ", ".join(sorted({d for _, d in generators})),
          "generator_licence": ", ".join(sorted({synth.generator_licence(g) for g, _ in generators})),
          "gpus": GPU_NAMES if todo else None, "code_sha": CODE_SHA, "data_revision": DATA_REVISION,
          "gen_seconds": round(sum(r["gen_seconds"] for r in rows), 1),
          "new_tokens": sum(r["new_tokens"] for r in rows)})
synth.validate_stats(stats)
uploads = {"synth/filtered.ndjson": "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in kept).encode(),
           "synth/synth_stats.json": (json.dumps(stats, indent=2) + "\n").encode()}
if PROMOTE:
    uploads["synth/synth_chats.jsonl"] = uploads["synth/filtered.ndjson"]
for dest, body in uploads.items():
    api.upload_file(path_or_fileobj=body, path_in_repo=dest, repo_id=DATA_REPO, repo_type="dataset",
                    commit_message=f"synth filtered: {len(kept)}/{len(rows)} kept")
with open(f"{OUTPUT}/synth_stats.json", "wb") as f:  # counts only (Kaggle: kernels output)
    f.write(uploads["synth/synth_stats.json"])
print("SYNTH_STATS", json.dumps(stats))